# EDA-Based Feature Engineering And Retraining

Goal: after baseline model comparison, create new features from EDA insights and retrain the selected best models.

Workflow:
1. Load clean common data.
2. Split into train/validation/test.
3. Fit feature-engineering thresholds on train only.
4. Apply the same feature engineering to train/validation/test.
5. Preprocess again with scaler and one-hot encoder.
6. Retrain Random Forest and SVM.
7. Select the best retrained model using validation metrics.

Important: the test set is created and processed, but not used for model selection.

In [1]:
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == "feature_engineering" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "diabetic_data_clean_common.csv"
OUTPUT_DIR = ROOT_DIR / "train" / "feature_engineering" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "readmitted_binary"
RANDOM_STATE = 42

DATA_PATH, OUTPUT_DIR

(WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/data/diabetic_data_clean_common.csv'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/feature_engineering/eda_outputs'))

## 1. Load Clean Common Data

This file is still close to the original dataset. It already contains common cleaned fields from EDA/preprocessing such as `readmitted_binary`, `age_ordinal`, and diagnosis groups.

In [2]:
df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
print("Target distribution:")
print(df[TARGET_COLUMN].value_counts(normalize=True).rename("ratio"))

df.head()

Data shape: (101763, 51)
Target distribution:
readmitted_binary
0    0.539106
1    0.460894
Name: ratio, dtype: float64


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_binary,age_midpoint,age_ordinal,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,NO,0,5,0,Endocrine_Metabolic,Unknown,Unknown
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,Ch,Yes,>30,1,15,1,Endocrine_Metabolic,Endocrine_Metabolic,Endocrine_Metabolic
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,No,No,Yes,NO,0,25,2,Pregnancy_Childbirth,Endocrine_Metabolic,Supplementary_V
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,Ch,Yes,NO,0,35,3,Infectious_Parasitic,Endocrine_Metabolic,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,No,Ch,Yes,NO,0,45,4,Neoplasms,Neoplasms,Endocrine_Metabolic


## 2. Split First

We split before fitting thresholds such as Q3. This avoids data leakage from validation/test into train.

In [3]:
y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (65128, 50)
X_val: (16282, 50)
X_test: (20353, 50)


## 3. EDA-Based Feature Plan

Features created from EDA insights:

- Data quality: missing indicators for race, payer code, medical specialty, and diagnoses.
- Admission/demographic EDA: long stay, short stay, senior patient, emergency/urgent/elective admission, discharge-risk flags.
- Medical EDA: prior visit totals, per-day treatment intensity, high-medication/high-lab/high-diagnosis indicators.
- A1C/glucose EDA: tested/abnormal flags.
- Drug EDA: insulin usage/change, number of diabetes drugs used, number of drugs changed.
- Diagnosis EDA: same diagnosis group flags and number of unique diagnosis groups.

In [4]:
DRUG_COLUMNS = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan).fillna(0)


def fit_feature_engineering_params(X_train):
    return {
        "q3_time_in_hospital": X_train["time_in_hospital"].quantile(0.75),
        "q3_num_medications": X_train["num_medications"].quantile(0.75),
        "q3_num_lab_procedures": X_train["num_lab_procedures"].quantile(0.75),
        "q3_number_diagnoses": X_train["number_diagnoses"].quantile(0.75),
    }


def add_eda_features(X, params):
    X_fe = X.copy()

    # Data quality features from EDA 01.
    X_fe["race_unknown"] = X_fe["race"].isna().astype(int)
    X_fe["payer_code_unknown"] = X_fe["payer_code"].isna().astype(int)
    X_fe["medical_specialty_unknown"] = X_fe["medical_specialty"].isna().astype(int)
    X_fe["diag_missing_count"] = X_fe[["diag_1", "diag_2", "diag_3"]].isna().sum(axis=1)

    # Demographic/admission features from EDA 02.
    X_fe["is_senior"] = (X_fe["age_ordinal"] >= 6).astype(int)
    X_fe["is_long_stay_fixed"] = (X_fe["time_in_hospital"] >= 7).astype(int)
    X_fe["is_long_stay_q3"] = (X_fe["time_in_hospital"] >= params["q3_time_in_hospital"]).astype(int)
    X_fe["is_short_stay"] = (X_fe["time_in_hospital"] <= 2).astype(int)
    X_fe["is_emergency_admission"] = X_fe["admission_type_id"].isin([1]).astype(int)
    X_fe["is_urgent_admission"] = X_fe["admission_type_id"].isin([2]).astype(int)
    X_fe["is_elective_admission"] = X_fe["admission_type_id"].isin([3]).astype(int)
    X_fe["is_from_emergency_room"] = X_fe["admission_source_id"].isin([7]).astype(int)
    X_fe["is_discharged_home"] = X_fe["discharge_disposition_id"].isin([1]).astype(int)
    X_fe["is_transferred"] = X_fe["discharge_disposition_id"].isin([2, 3, 4, 5, 6, 22, 23, 24]).astype(int)
    X_fe["is_high_risk_discharge"] = X_fe["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21]).astype(int)

    # Medical/treatment intensity features from EDA 03.
    X_fe["total_prior_visits"] = X_fe["number_outpatient"] + X_fe["number_emergency"] + X_fe["number_inpatient"]
    X_fe["has_prior_inpatient"] = (X_fe["number_inpatient"] > 0).astype(int)
    X_fe["has_emergency_visit"] = (X_fe["number_emergency"] > 0).astype(int)
    X_fe["has_outpatient_visit"] = (X_fe["number_outpatient"] > 0).astype(int)
    X_fe["labs_per_day"] = safe_divide(X_fe["num_lab_procedures"], X_fe["time_in_hospital"])
    X_fe["meds_per_day"] = safe_divide(X_fe["num_medications"], X_fe["time_in_hospital"])
    X_fe["procedures_per_day"] = safe_divide(X_fe["num_procedures"], X_fe["time_in_hospital"])
    X_fe["diagnoses_per_day"] = safe_divide(X_fe["number_diagnoses"], X_fe["time_in_hospital"])
    X_fe["meds_per_diagnosis"] = safe_divide(X_fe["num_medications"], X_fe["number_diagnoses"])
    X_fe["high_num_medications"] = (X_fe["num_medications"] >= params["q3_num_medications"]).astype(int)
    X_fe["high_num_lab_procedures"] = (X_fe["num_lab_procedures"] >= params["q3_num_lab_procedures"]).astype(int)
    X_fe["high_number_diagnoses"] = (X_fe["number_diagnoses"] >= params["q3_number_diagnoses"]).astype(int)
    X_fe["treatment_complexity"] = (
        X_fe["num_lab_procedures"]
        + X_fe["num_procedures"]
        + X_fe["num_medications"]
        + X_fe["number_diagnoses"]
    )

    # A1C/glucose features from EDA 03.
    a1c = X_fe["A1Cresult"].fillna("None")
    glu = X_fe["max_glu_serum"].fillna("None")
    X_fe["a1c_tested"] = (~a1c.isin(["None", "nan"])).astype(int)
    X_fe["a1c_abnormal"] = a1c.isin([">7", ">8"]).astype(int)
    X_fe["glu_tested"] = (~glu.isin(["None", "nan"])).astype(int)
    X_fe["glu_abnormal"] = glu.isin([">200", ">300"]).astype(int)

    # Drug features from EDA 03.
    drug_data = X_fe[DRUG_COLUMNS].fillna("No")
    X_fe["insulin_used"] = (~X_fe["insulin"].fillna("No").eq("No")).astype(int)
    X_fe["insulin_changed"] = X_fe["insulin"].isin(["Up", "Down"]).astype(int)
    X_fe["medication_changed"] = X_fe["change"].eq("Ch").astype(int)
    X_fe["diabetes_med_used"] = X_fe["diabetesMed"].eq("Yes").astype(int)
    X_fe["num_diabetes_drugs_used"] = drug_data.ne("No").sum(axis=1)
    X_fe["num_drugs_changed"] = drug_data.isin(["Up", "Down"]).sum(axis=1)
    X_fe["any_drug_up"] = drug_data.eq("Up").any(axis=1).astype(int)
    X_fe["any_drug_down"] = drug_data.eq("Down").any(axis=1).astype(int)

    # Diagnosis group features from EDA 03.
    diag_groups = X_fe[["diag_1_group", "diag_2_group", "diag_3_group"]].fillna("Unknown")
    X_fe["same_diag_1_2_group"] = diag_groups["diag_1_group"].eq(diag_groups["diag_2_group"]).astype(int)
    X_fe["same_diag_1_3_group"] = diag_groups["diag_1_group"].eq(diag_groups["diag_3_group"]).astype(int)
    X_fe["same_diag_2_3_group"] = diag_groups["diag_2_group"].eq(diag_groups["diag_3_group"]).astype(int)
    X_fe["num_unique_diag_groups"] = diag_groups.nunique(axis=1)

    return X_fe

In [5]:
fe_params = fit_feature_engineering_params(X_train)
fe_params

{'q3_time_in_hospital': np.float64(6.0),
 'q3_num_medications': np.float64(20.0),
 'q3_num_lab_procedures': np.float64(57.0),
 'q3_number_diagnoses': np.float64(9.0)}

In [6]:
X_train_fe = add_eda_features(X_train, fe_params)
X_val_fe = add_eda_features(X_val, fe_params)
X_test_fe = add_eda_features(X_test, fe_params)

new_features = sorted(set(X_train_fe.columns) - set(X_train.columns))

print("Original feature count:", X_train.shape[1])
print("New EDA feature count:", len(new_features))
print("Feature count after EDA FE:", X_train_fe.shape[1])

pd.DataFrame({"new_feature": new_features})

Original feature count: 50
New EDA feature count: 44
Feature count after EDA FE: 94


,new_feature
0,a1c_abnormal
1,a1c_tested
2,any_drug_down
3,any_drug_up
4,diabetes_med_used
5,diag_missing_count
6,diagnoses_per_day
7,glu_abnormal
8,glu_tested
9,has_emergency_visit


## 4. Prepare Preprocessing

Drop raw high-cardinality diagnosis codes and raw target text. Keep diagnosis groups. Numeric features are scaled. Categorical features are one-hot encoded.

In [7]:
DROP_COLUMNS = [
    "readmitted",
    "diag_1",
    "diag_2",
    "diag_3",
    "age",
    "age_midpoint",
]

X_train_model = X_train_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_train_fe.columns])
X_val_model = X_val_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_val_fe.columns])
X_test_model = X_test_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_test_fe.columns])

categorical_cols = X_train_model.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X_train_model.columns if col not in categorical_cols]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Total model input columns before encoding:", X_train_model.shape[1])

Numeric columns: 56
Categorical columns: 32
Total model input columns before encoding: 88


In [8]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("onehot", make_one_hot_encoder())]), categorical_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_model)
X_val_processed = preprocessor.transform(X_val_model)
X_test_processed = preprocessor.transform(X_test_model)

feature_names = preprocessor.get_feature_names_out()

print("Processed X_train:", X_train_processed.shape)
print("Processed X_val:", X_val_processed.shape)
print("Processed X_test:", X_test_processed.shape)

Processed X_train: (65128, 287)
Processed X_val: (16282, 287)
Processed X_test: (20353, 287)


## 5. Retrain Best Baseline Models

From the baseline comparison, the practical top models are:
- Random Forest
- SVM using LinearSVC

In [9]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "SVM": LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=8000,
        random_state=RANDOM_STATE,
    ),
}


def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate(name, model, X_train_processed, y_train, X_val_processed, y_val):
    start_time = time.time()
    model.fit(X_train_processed, y_train)
    train_time = time.time() - start_time

    y_pred = model.predict(X_val_processed)
    y_score = get_score_for_auc(model, X_val_processed)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1_score": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_score) if y_score is not None else None,
        "train_time_sec": train_time,
    }
    cm = pd.DataFrame(
        confusion_matrix(y_val, y_pred),
        index=["actual_0", "actual_1"],
        columns=["predicted_0", "predicted_1"],
    )
    report = pd.DataFrame(classification_report(y_val, y_pred, output_dict=True, zero_division=0)).T
    return metrics, cm, report, model

In [10]:
results = []
confusion_matrices = {}
classification_reports = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name} with EDA features...")
    metrics, cm, report, trained_model = evaluate(
        name,
        model,
        X_train_processed,
        y_train,
        X_val_processed,
        y_val,
    )
    results.append(metrics)
    confusion_matrices[name] = cm
    classification_reports[name] = report
    trained_models[name] = trained_model
    print(
        f"Done {name}: "
        f"F1={metrics['f1_score']:.4f}, "
        f"Recall={metrics['recall']:.4f}, "
        f"Precision={metrics['precision']:.4f}, "
        f"Accuracy={metrics['accuracy']:.4f}, "
        f"ROC-AUC={metrics['roc_auc']:.4f}"
    )

eda_retrain_results_df = pd.DataFrame(results).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
).reset_index(drop=True)

eda_retrain_results_df

Training Random Forest with EDA features...
Done Random Forest: F1=0.6090, Recall=0.6022, Precision=0.6160, Accuracy=0.6437, ROC-AUC=0.7006
Training SVM with EDA features...
Done SVM: F1=0.6066, Recall=0.6070, Precision=0.6063, Accuracy=0.6372, ROC-AUC=0.6908


,model,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,0.643656,0.616003,0.602212,0.609030,0.700582,6.785921
1,SVM,0.637207,0.606282,0.607010,0.606646,0.690755,17.415432


## 6. Select Best Retrained Model On Validation

In [11]:
best_model_name = eda_retrain_results_df.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Best model after EDA feature engineering:", best_model_name)
display(eda_retrain_results_df.head(1))

print("Validation confusion matrix:")
display(confusion_matrices[best_model_name])

print("Validation classification report:")
display(classification_reports[best_model_name])

Best model after EDA feature engineering: Random Forest


,model,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,0.643656,0.616003,0.602212,0.60903,0.700582,6.785921


Validation confusion matrix:


,predicted_0,predicted_1
actual_0,5961,2817
actual_1,2985,4519


Validation classification report:


,precision,recall,f1-score,support
0,0.666331,0.679084,0.672647,8778.000000
1,0.616003,0.602212,0.609030,7504.000000
accuracy,0.643656,0.643656,0.643656,0.643656
macro avg,0.641167,0.640648,0.640838,16282.000000
weighted avg,0.643136,0.643656,0.643327,16282.000000


## 7. Save Outputs

These outputs can be used for final testing or reporting.

In [12]:
train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
val_processed_df = pd.DataFrame(X_val_processed, columns=feature_names)
test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

train_processed_df[TARGET_COLUMN] = y_train.to_numpy()
val_processed_df[TARGET_COLUMN] = y_val.to_numpy()
test_processed_df[TARGET_COLUMN] = y_test.to_numpy()

train_processed_df.to_csv(OUTPUT_DIR / "eda_train_processed.csv", index=False)
val_processed_df.to_csv(OUTPUT_DIR / "eda_validation_processed.csv", index=False)
test_processed_df.to_csv(OUTPUT_DIR / "eda_test_processed.csv", index=False)

pd.Series(feature_names, name="feature_name").to_csv(OUTPUT_DIR / "eda_feature_names.csv", index=False)
pd.DataFrame({"new_feature": new_features}).to_csv(OUTPUT_DIR / "eda_new_features.csv", index=False)
pd.DataFrame([fe_params]).to_csv(OUTPUT_DIR / "eda_feature_engineering_params.csv", index=False)
eda_retrain_results_df.to_csv(OUTPUT_DIR / "eda_retrain_validation_metrics.csv", index=False)

for name, cm in confusion_matrices.items():
    safe_name = name.lower().replace(" ", "_")
    cm.to_csv(OUTPUT_DIR / f"{safe_name}_validation_confusion_matrix.csv")
    classification_reports[name].to_csv(OUTPUT_DIR / f"{safe_name}_validation_classification_report.csv")

joblib.dump(preprocessor, OUTPUT_DIR / "eda_preprocessor.joblib")
joblib.dump(best_model, OUTPUT_DIR / "eda_best_validation_model.joblib")

print(f"Saved outputs to: {OUTPUT_DIR}")

Saved outputs to: d:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering\eda_outputs
